In [1]:
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

from katabatic.models.tabddpm.models import Tabddpm

ROOT set to: C:\Users\Prabu\Downloads\Katabatic


In [2]:
# Preprocess data
dataset_path = ROOT / "raw_data" / "adult.csv"         
output_path = ROOT / "discretized_data" / "adult.csv"  

output_path.parent.mkdir(parents=True, exist_ok=True)

# Preprocess
discretize_preprocess(str(dataset_path), str(output_path))

Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\adult.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\adult.csv


In [11]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)         
warnings.filterwarnings("ignore", message="Parameters: {")           
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")
input_csv = str(output_path)
output_dir = str(ROOT / "sample_data" / "adult")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "adult" / "tabddpm")

class TabDDPMFull(Tabddpm):
    def train(self, *args, **kwargs):
       
        config = kwargs.get("config") or {}
        if not isinstance(config, dict):
            config = {}

      
        config["steps"] = 8000

        # Pass updated config into the original train()
        kwargs["config"] = config
        return super().train(*args, **kwargs)

pipeline = TrainTestSplitPipeline(
    model=lambda: TabDDPMFull()
)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print(result)



Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Step 100/8000 | MLoss: 0.0000 | GLoss: 1.2456
Step 200/8000 | MLoss: 0.0000 | GLoss: 0.4868
Step 300/8000 | MLoss: 0.0000 | GLoss: 0.1440
Step 400/8000 | MLoss: 0.0000 | GLoss: 0.0641
Step 500/8000 | MLoss: 0.0000 | GLoss: 0.0173
Step 600/8000 | MLoss: 0.0000 | GLoss: 0.0146
Step 700/8000 | MLoss: 0.0000 | GLoss: 0.0182
Step 800/8000 | MLoss: 0.0000 | GLoss: 0.0149
Step 900/8000 | MLoss: 0.0000 | GLoss: 0.0124
Step 1000/8000 | MLoss: 0.0000 | GLoss: 0.0241
Step 1100/8000 | MLoss: 0.0000 | GLoss: 0.0051
Step 1200/8000 | MLoss: 0.0000 | GLoss: 0.0043
Step 1300/8000 | MLoss: 0.0000 | GLoss: 0.0148
Step 1400/8000 | 